## Train a K-Means Clustering Model
The model will be trained using pandas and scikit-learn.
The model will be trained from data found at https://www.kaggle.com/datasets/itssuru/loan-data

I am running this through VS Code, using a docker container. You may also use the `ipynb` file in other Jupyter Notebook style setups. Consult Jupyter Notebook for options.

**To download the data, run this cell.**

Running this cell will download the data, if you are running it in the docker container. If not, you will need to navigate to the `KAGGLE_DATA_URL` and download the data manually.

In [10]:
import os
from data import download_kaggle_dataset
KAGGLE_DATA_URL = "https://www.kaggle.com/datasets/itssuru/loan-data"
DATA_PATH = os.path.join(os.getcwd(), "data", "k_means")
download_kaggle_dataset(KAGGLE_DATA_URL, DATA_PATH)

/workspaces/MS365/src/data/k_means contains data. Delete the file(s) if you want to download again.


**Import the necessary python packages**

Import `pandas`, `numpy`, `sklearn.preprocessing.MinMaxScaler`, `sklearn.preprocessing.OneHotEncoder`, `sklearn.cluster.KMeans`, `sklearn.metrics.pairwise_distances`,  `sklearn.decomposition.TruncatedSVD`, `sklearn.decomposition.PCA`, and `matplotlib.pyplot`. Typically, packages such as `pandas`, `numpy`, and `matplotlib.pyplot` are imported with an allias. I will not be following that strategy here. 

The default size of the plots from `matplotlib.pyplot` is 6.4 inches by 4.8 inches ([width by height](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.figure.html)). I wll be setting them to 20 inches by 5 inches. This will create a more readable output. Depending on your screen size, you may want to change this for your own use. 

By default, `pandas` will truncate datasets with a lot of rows and a lot of columns. You can alter this functionality with the `set_option()` function. I have set it to show all possible columns. This could result in long run times for cells where you are displaying the data, if there are many columns to display. This will be expected behavior for this analysis.

If you are running the docker container or if you are using [Google Colab](https://colab.research.google.com/), the `pip install` has already been done. If not, then please consult your jupyter notebook environment docs for how to install the needed packages.

In [47]:
import pandas
import numpy
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import TruncatedSVD, PCA
import matplotlib.pyplot

matplotlib.pyplot.rcParams["figure.figsize"] = [20, 5]
pandas.set_option("display.max_columns", None)

**Import and Review Data**

Import the data downloaded from the Kaggle site. The file name is `loan_data.csv`. Import it into `df` using `pandas.read_csv()`. The file is a typical CSV file, separated by commas.

The column names contain dots to separate words. Change those to underscores using `df.columns.str.replace()`. Underscores tend to be a common strategy for separating words in a column header.

Display the top rows of the dataframe using the `head()` method. The default value is `n=5`. Change the value if you want to see more or less rows.

In [48]:
df = pandas.read_csv(os.path.join(DATA_PATH, "loan_data.csv"))
df.columns = df.columns.str.replace(".", "_")
df.head(n=15)

,credit_policy,purpose,int_rate,installment,log_annual_inc,dti,fico,days_with_cr_line,revol_bal,revol_util,inq_last_6mths,delinq_2yrs,pub_rec,not_fully_paid
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0
5,1,credit_card,0.0788,125.13,11.904968,16.98,727,6120.041667,50807,51.0,0,0,0,0
6,1,debt_consolidation,0.1496,194.02,10.714418,4.00,667,3180.041667,3839,76.8,0,0,1,1
7,1,all_other,0.1114,131.22,11.002100,11.08,722,5116.000000,24220,68.6,0,0,0,1
8,1,home_improvement,0.1134,87.19,11.407565,17.25,682,3989.000000,69909,51.1,1,0,0,0
9,1,debt_consolidation,0.1221,84.12,10.203592,10.00,707,2730.041667,5630,23.0,1,0,0,0


The top 15 rows of the data show the type of data that will be used to train the K-Means Clustering model. The model will require numerical data. The only column that will not work for our analysis is `purpose`. The data can be removed from the dataframe, or it can be encoded. Given that it is nominal categorical data, the best option will be to one-hot encode the data.

The other concern is if there are any missing values. Running [`describe()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) on the data will show the counts for each of the columns. The default output of the describe method will typically only show numerical values. To get a count for all the columns, the `include="all"` parameter can be used. Reviewing the count for each column, the data show that all columns contain the same number of counts. This means there are not any missing values in any of the columns. Nothing needs to be done to correct for missing data.

In [49]:
df.describe(include="all")

,credit_policy,purpose,int_rate,installment,log_annual_inc,dti,fico,days_with_cr_line,revol_bal,revol_util,inq_last_6mths,delinq_2yrs,pub_rec,not_fully_paid
count,9578.000000,9578,9578.000000,9578.000000,9578.000000,9578.000000,9578.000000,9578.000000,9.578000e+03,9578.000000,9578.000000,9578.000000,9578.000000,9578.000000
unique,NaN,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,debt_consolidation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,3957,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,0.804970,NaN,0.122640,319.089413,10.932117,12.606679,710.846314,4560.767197,1.691396e+04,46.799236,1.577469,0.163708,0.062122,0.160054
std,0.396245,NaN,0.026847,207.071301,0.614813,6.883970,37.970537,2496.930377,3.375619e+04,29.014417,2.200245,0.546215,0.262126,0.366676
min,0.000000,NaN,0.060000,15.670000,7.547502,0.000000,612.000000,178.958333,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,NaN,0.103900,163.770000,10.558414,7.212500,682.000000,2820.000000,3.187000e+03,22.600000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,NaN,0.122100,268.950000,10.928884,12.665000,707.000000,4139.958333,8.596000e+03,46.300000,1.000000,0.000000,0.000000,0.000000
75%,1.000000,NaN,0.140700,432.762500,11.291293,17.950000,737.000000,5730.000000,1.824950e+04,70.900000,2.000000,0.000000,0.000000,0.000000


**Clean the Data**

The data need to be numerical to work with the K-Means Clustering model. The column `purpose` will need to be one-hot encoded to be useful for this analysis. Create a one-hot encoder entity using [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html); include the parameter `sparse_output=False`. Assign the one-hot encoder to `ohe`. Review the documentation for `OneHotEncoder` if you want to better understand the different parameters that can be used and their function.

Use the `fit_transform` method to convert the data from the `purpose` column to one-hot encoded columns of data. The `fit_transform` requires an array-like object with n number of samples and n number of features. This can be resolved by passing in a dataframe object. Create a dataframe object from the `purpose` column by wrapping the column name by two square brackets `[[purpose]]`. Pass this value in to the `ohe.fit_transform` method. The `fit_transform` method will return an array of arrays filled with 0s and 1s. Save this information to `purpose_encoded`.

Create a pandas DataFrame from the `purpose_encoded` data and the `ohe.get_feature_names_out()` method. Pass those two values in to the `pandas.DataFrame` method. This will create a dataframe with the needed data.

Merge the new `purpose_df` dataframe with the `df` dataframe created earlier. Merge the two dataframes on the indexes using [`pandas.merge`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html). Review the documentation to better understand the parameters and their uses. Save the result of the merge to `df`, overwriting the original `df` value created earlier.

In [50]:
ohe = OneHotEncoder(sparse_output=False)
purpose_encoded = ohe.fit_transform(df[["purpose"]])
purpose_df = pandas.DataFrame(purpose_encoded, columns=ohe.get_feature_names_out())
df = pandas.merge(df, purpose_df, left_index=True, right_index=True)

In [52]:
df.head()
# min_max = MinMaxScaler()
# scaled_data = min_max.fit_transform(df)
# scaled_df = pd.DataFrame(scaled_data, columns=min_max.get_feature_names_out())
# scaled_df.head()

,credit_policy,purpose,int_rate,installment,log_annual_inc,dti,fico,days_with_cr_line,revol_bal,revol_util,inq_last_6mths,delinq_2yrs,pub_rec,not_fully_paid,purpose_all_other,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_major_purchase,purpose_small_business
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
